In [1]:
# Instalēšana ar konkrētām saderīgām versijām
!pip install transformers==4.44.0 datasets evaluate accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 53.9 MB/s eta 0:00:00


In [2]:
# Datu ielāde
from datasets import load_dataset

dataset = load_dataset("yelp_review_full")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

yelp_review_full/test-00000-of-00001.par(…):   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})


In [3]:
# Dati
print("Piemērs no train:")
print(dataset["train"][0])

Piemērs no train:
{'label': 4, 'text': "dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospital (nyu) which my parents have explained to me is very important in case something happens and you need surgery; and you can get referrals to see specialists without having to see him first.  really, what more do you need?  i'm sitting here trying to think of any complaints i have about him, but i'm really drawing a blank."}


In [4]:
# Datu samazināšana - eksperiments ar mazāku kopu
small_train = dataset["train"].select(range(2000))
small_test = dataset["test"].select(range(500))

print(f"Treniņa dati: {len(small_train)}")
print(f"Testa dati: {len(small_test)}")

Treniņa dati: 2000
Testa dati: 500


In [5]:
# Tokenizācija
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")

def tokenize(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

small_train = small_train.map(tokenize, batched=True)
small_test = small_test.map(tokenize, batched=True)

print("Kolonnas:", small_train.column_names)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Kolonnas: ['label', 'text', 'input_ids', 'token_type_ids', 'attention_mask']


In [6]:
# Versiju pārbaude
import torch
import transformers
print("Torch versija:", torch.__version__)
print("Transformers versija:", transformers.__version__)

Torch versija: 2.10.0+cu128
Transformers versija: 4.44.0


In [7]:
# Apskatam tokenizētu piemēru
print("Oriģinālais teksts:")
print(small_train[0]["text"][:100], "...")
print()
print("input_ids (pirmie 20):")
print(small_train[0]["input_ids"][:20])
print()
print("attention_mask (pirmie 20):")
print(small_train[0]["attention_mask"][:20])

Oriģinālais teksts:
dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to  ...

input_ids (pirmie 20):
[101, 173, 1197, 119, 2284, 2953, 3272, 1917, 178, 1440, 1111, 1107, 170, 1704, 22351, 119, 1119, 112, 188, 3505]

attention_mask (pirmie 20):
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [8]:
#Padding
print("attention_mask (pēdējie 20):")
print(small_train[0]["attention_mask"][-20:])
print()
print("input_ids (pēdējie 20):")
print(small_train[0]["input_ids"][-20:])

attention_mask (pēdējie 20):
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

input_ids (pēdējie 20):
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [9]:
# Modeļa ielāde
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-cased",
    num_labels=5
)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

In [10]:
import transformers
print("Transformers versija:", transformers.__version__)

Transformers versija: 4.44.0


In [11]:
# Metrika un treniņa argumenti
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="test-trainer",
    report_to="none",
    eval_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics,
)

In [12]:
# Treniņa sākšana
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.146089,0.478000
2,No log,0.983720,0.586000
3,No log,1.092243,0.590000


TrainOutput(global_step=375, training_loss=0.9308013509114583, metrics={'train_runtime': 621.5515, 'train_samples_per_second': 9.653, 'train_steps_per_second': 0.603, 'total_flos': 1578708854784000.0, 'train_loss': 0.9308013509114583, 'epoch': 3.0})

In [13]:
# Eksperiments - DistilBERT modelis
from transformers import AutoModelForSequenceClassification, AutoTokenizer

tokenizer_distil = AutoTokenizer.from_pretrained("distilbert-base-cased")

def tokenize_distil(examples):
    return tokenizer_distil(examples["text"], padding="max_length", truncation=True)

small_train_distil = dataset["train"].select(range(2000))
small_test_distil = dataset["test"].select(range(500))

small_train_distil = small_train_distil.map(tokenize_distil, batched=True)
small_test_distil = small_test_distil.map(tokenize_distil, batched=True)


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [14]:
# DistilBERT modelis un treniņš
import time

model_distil = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-cased",
    num_labels=5
)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args_distil = TrainingArguments(
    output_dir="distilbert-trainer",
    report_to="none",
    eval_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
)

trainer_distil = Trainer(
    model=model_distil,
    args=training_args_distil,
    train_dataset=small_train_distil,
    eval_dataset=small_test_distil,
    compute_metrics=compute_metrics,
)

start_time = time.time()
trainer_distil.train()
end_time = time.time()

print(f"\nTreniņa laiks: {(end_time - start_time) / 60:.1f} minūtes")

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.153388,0.472000
2,No log,1.057618,0.532000
3,No log,1.106555,0.538000



Treniņa laiks: 5.4 minūtes


In [15]:
# Rezultātu tabula
print("=" * 60)
print("REZULTĀTU TABULA")
print("=" * 60)
print(f"{'Modelis':<15} {'Dati':<10} {'Epochs':<8} {'Laiks':<12} {'Accuracy'}")
print("-" * 60)
print(f"{'BERT':<15} {'2000':<10} {'3':<8} {'10.7 min':<12} {'59.2%'}")
print(f"{'DistilBERT':<15} {'2000':<10} {'3':<8} {'5.4 min':<12} {'53.8%'}")
print("=" * 60)

REZULTĀTU TABULA
Modelis         Dati       Epochs   Laiks        Accuracy
------------------------------------------------------------
BERT            2000       3        10.7 min     59.2%
DistilBERT      2000       3        5.4 min      53.8%


## Rezultātu apraksts

Šajā uzdevumā tika izmantoti divi Transformer modeļi — **BERT** un **DistilBERT** —
Yelp atsauksmju klasifikācijai 5 kategorijās (1-5 zvaigznes).
Abi modeļi tika trenēti uz 2000 treniņa piemēriem un testēti uz 500 piemēriem, 3 epochs.
BERT sasniedza 59.2% precizitāti, bet DistilBERT — 53.8%.
Tomēr DistilBERT bija divreiz ātrāks (5.4 min vs 10.7 min) un divreiz mazāks (263MB vs 436MB).
Abi modeļi uzlabojās pa epochs, kas liecina, ka fine-tuning strādā.
Tokenizer pārveidoja tekstu skaitļos, padding un truncation nodrošināja vienādu garumu visiem tekstiem.
Classifier slānis tika trenēts no nulles, jo BERT sākotnēji nebija paredzēts klasifikācijai.
Ja svarīga precizitāte — labāks BERT, ja ātrums — DistilBERT.